<a href="https://colab.research.google.com/github/Itzmepromgitman/downloadere/blob/main/animepahe_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<blockquote><span style="font-family: 'Comic Sans MS', cursive; font-size: 24px; color: #8A2BE2;">😈 The AnimePahe Soul Harvester (Colab Edition) 🦇</span></blockquote>

**A sinfully fast, elegant notebook to devour episodes from AnimePahe, merge them into your collection, and banish them to the cloud.**
---
## 📜 DISCLAIMER
Personal / educational use only.
Respect copyright laws. Do NOT use this notebook for commercial or infringing purposes.

In [ ]:
from IPython.display import HTML, display
def print_bot(msg):
    display(HTML(f"<blockquote><span style='font-family: \"Comic Sans MS\", cursive; font-size: 14px; color: #ff007f;'>😈 {msg} 🦇</span></blockquote>"))

print_bot("Summoning the dark dependencies... 🪄✨ Please wait while I weave the magic...")
!pip install -q requests beautifulsoup4 cloudscraper m3u8 pycryptodome tqdm yt-dlp > /dev/null 2>&1
!apt-get -qq install -y ffmpeg aria2 nodejs jq fzf curl > /dev/null 2>&1
!git clone https://github.com/KevCui/animepahe-dl.git > /dev/null 2>&1
!chmod +x animepahe-dl/animepahe-dl.sh
print_bot("✅ Dark pact sealed! Dependencies installed and bash demons chained! 🩸")

In [ ]:
from IPython.display import HTML, display
def print_bot(msg):
    display(HTML(f"<blockquote><span style='font-family: \"Comic Sans MS\", cursive; font-size: 14px; color: #ff007f;'>😈 {msg} 🦇</span></blockquote>"))

# @title ⚙️ **The Devil's Configuration** { display-mode: "form" }

#@markdown ### 🔗 AnimePahe Slug or ID
#@markdown (e.g., `frieren-beyond-journeys-end` or the raw UUID from the URL)
anime_slug = "frieren-beyond-journeys-end"  # @param {type:"string"}

#@markdown ### 📺 Episode Selection
download_mode = "Single Episode"  # @param ["All Episodes", "Episode Range", "Single Episode"]
single_episode = "1"  # @param {type:"string"}
start_episode = "1"  # @param {type:"string"}
end_episode   = "2"  # @param {type:"string"}

#@markdown ### 🎥 Quality & Audio Curses
video_quality = "1080p"  # @param ["1080p", "720p", "480p", "360p"]
prefer_audio = "jpn"  # @param ["jpn", "eng"]

#@markdown ### 📥 Download Sorcery
download_method = "yt-dlp"  # @param ["yt-dlp"]
max_workers = 20    # @param {type:"slider", min:1, max:32, step:1}
max_retries = 10    # @param {type:"slider", min:1, max:10, step:1}
connection_timeout = 600  # @param {type:"slider", min:60, max:600, step:30}

#@markdown ### 🔗 Merge Ritual
merge_episodes = False  # @param {type:"boolean"}
season_number  = 1     # @param {type:"integer"}
keep_individual_files = False  # @param {type:"boolean"}

#@markdown ### 📤 Banishment Destination
upload_destination = "None (Keep Local)"  # @param ["GoFile.io Only", "Google Drive Only", "Both Services", "None (Keep Local)"]
upload_merged_only = True    # @param {type:"boolean"}

print_bot("Configuration bound to the scroll! 📜✨ Ready to execute the heist! 😈")


In [ ]:
import requests
import re
import json
import os
import time
import subprocess
from typing import List, Optional, Tuple, Dict, Any
import cloudscraper
import shutil
from tqdm.notebook import tqdm

from IPython.display import HTML, display
def print_bot(msg):
    display(HTML(f"<blockquote><span style='font-family: \"Comic Sans MS\", cursive; font-size: 14px; color: #ff007f;'>😈 {msg} 🦇</span></blockquote>"))

BASE_URL = "https://animepahe.pw"
API_URL = f"{BASE_URL}/api"
REFERER_URL = "https://kwik.cx/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": BASE_URL,
}

scraper = cloudscraper.create_scraper(browser={"browser": "chrome", "platform": "windows", "desktop": True})

# ---- ANIMEPAHE LOGIC ----

def get_episodes(slug: str):
    print_bot(f"Searching the underworld for episodes of `{slug}`...")
    page = 1
    episodes = []
    while True:
        url = f"{API_URL}?m=release&id={slug}&sort=episode_asc&page={page}"
        r = scraper.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            break
        data = r.json()
        if 'data' not in data:
            break
        episodes.extend(data['data'])
        if page >= data.get('last_page', 1):
            break
        page += 1
    return episodes

def get_m3u8_link(slug: str, ep_num: str):
    print_bot(f"Summoning the bash demons to extract link for Episode {ep_num}... 🕸️")
    cmd = [
        "bash", "animepahe-dl/animepahe-dl.sh", 
        "-s", slug, 
        "-e", str(ep_num), 
        "-r", video_quality.replace("p", ""), 
        "-o", prefer_audio,
        "-l"
    ]
    try:
        out = subprocess.check_output(cmd, text=True, stderr=subprocess.DEVNULL)
        for line in out.splitlines():
            if line.startswith("http") and "m3u8" in line:
                return line
        return None
    except Exception as e:
        print_bot(f"The demons failed us: {e}")
        return None

# ---- DOWNLOAD LOGIC ----
def download_with_ytdlp(url: str, output_file: str, episode_label: str) -> bool:
    print_bot(f"Unleashing yt-dlp to drain Episode {episode_label}... 🧛‍♂️")
    cmd = [
        "yt-dlp",
        url,
        "-o", output_file,
        "--no-warnings",
        "--no-check-certificate",
        "--concurrent-fragments", str(max_workers),
        "--retries", str(max_retries),
        "--fragment-retries", str(max_retries),
        "--socket-timeout", str(connection_timeout),
        "--user-agent", HEADERS["User-Agent"],
        "--referer", REFERER_URL,
        "--newline",
    ]
    
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, bufsize=1
    )
    
    with tqdm(total=100, unit="%", desc=f"Ep {episode_label}") as pbar:
        last = 0.0
        for line in proc.stdout:
            m = re.search(r"\[download\]\s+(\d+\.?\d*)%", line)
            if m:
                cur = float(m.group(1))
                if cur > last:
                    pbar.update(cur - last)
                    last = cur
    proc.wait()
    if proc.returncode == 0 and os.path.exists(output_file):
        print_bot(f"Ahaha! Soul trapped in file: {os.path.basename(output_file)} 🩸")
        return True
    return False

# ---- MERGE LOGIC ----
def merge_videos(file_list: List[str], anime_title: str, season_num: int, first_ep: str, last_ep: str) -> Optional[str]:
    valid_files = [f for f in file_list if os.path.exists(f)]
    if not valid_files:
        return None
        
    merged_filename = f"{anime_title}_S{season_num:02d}_E{first_ep}-E{last_ep}.mp4"
    merged_filename = re.sub(r"[<>:\"/\\|?*']", "", merged_filename)
    merged_path = os.path.join(os.path.dirname(valid_files[0]), merged_filename)
    
    print_bot(f"Fusing the captured souls into the ultimate vessel: {merged_filename} 🏺🔥")
    list_file = "filelist_merge.txt"
    with open(list_file, "w", encoding="utf-8") as f:
        for vf in valid_files:
            safe_path = os.path.abspath(vf).replace("'", "'\\''")
            f.write(f"file '{safe_path}'\n")
            
    cmd = ["ffmpeg", "-f", "concat", "-safe", "0", "-i", list_file, "-map", "0", "-c:v", "copy", "-c:a", "copy", "-c:s", "copy", "-y", merged_path]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    proc.wait()
    
    try:
        os.remove(list_file)
    except Exception:
        pass
        
    if proc.returncode == 0 and os.path.exists(merged_path):
        print_bot(f"Fusion successful! 😈 The abomination breathes: {merged_filename}")
        return merged_path
    return None

# ---- UPLOAD LOGIC ----
def upload_to_gofile(filepath: str) -> Optional[str]:
    print_bot(f"Banishing {os.path.basename(filepath)} to the GoFile dimension... 🕳️")
    try:
        r = requests.get("https://api.gofile.io/servers", timeout=15)
        data = r.json()
        server = data['data']['servers'][0]['name']
        upload_url = f"https://{server}.gofile.io/contents/uploadfile"
        
        with open(filepath, "rb") as f:
            files = {"file": f}
            res = requests.post(upload_url, files=files)
            j = res.json()
            if j.get("status") == "ok":
                link = j["data"]["downloadPage"]
                print_bot(f"Hehehe... Trapped in the void! Link: <a href='{link}' style='color: #00ffff;'>{link}</a> 🔗")
                return link
    except Exception as e:
        print_bot(f"Curse it! The banishment failed: {e} 💥")
    return None

def upload_to_gdrive(filepath: str) -> Optional[str]:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print_bot("Invoking the Google Drive pact... 📜🖋️")
            drive.mount("/content/drive")
        
        dest_dir = "/content/drive/MyDrive/AnimePahe_Hoard/"
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = os.path.join(dest_dir, os.path.basename(filepath))
        print_bot(f"Smuggling {os.path.basename(filepath)} into Google Drive vault... 🏦")
        shutil.copy2(filepath, dest_path)
        print_bot(f"Stored safely in the dark archives! 📁")
        return dest_path
    except Exception as e:
        print_bot(f"Google Drive rejected the offering: {e} ⚔️")
    return None

# ---- MAIN EXECUTION ----
def main():
    print_bot("=========================================================")
    print_bot("🎬 INITIATING THE ANIMEPAHE SOUL HARVEST RITUAL 🦇")
    print_bot("=========================================================")
    
    # Extract slug if a full URL was provided
    global anime_slug
    if "animepahe." in anime_slug and "/anime/" in anime_slug:
        anime_slug = anime_slug.split("/anime/")[-1].strip("/")
    elif "animepahe." in anime_slug and "/play/" in anime_slug:
        anime_slug = anime_slug.split("/play/")[-1].split("/")[0].strip("/")
    
    episodes = get_episodes(anime_slug)
    if not episodes:
        print_bot(f"The prophecy is empty... No episodes found for {anime_slug}. Are you sure it exists? 🕸️")
        return
        
    selected = []
    if download_mode == "Single Episode":
        selected = [ep for ep in episodes if str(ep['episode']) == str(single_episode)]
    elif download_mode == "Episode Range":
        try:
            s = float(start_episode)
            e = float(end_episode)
            selected = [ep for ep in episodes if s <= float(ep['episode']) <= e]
        except Exception:
            pass
    else:
        selected = episodes

    if not selected:
        print_bot("Your demands yield nothing... No matching episodes! 💀")
        return

    print_bot(f"Target locked. I smell {len(selected)} souls ripe for the taking! 👁️👄👁️")
    
    download_dir = os.path.join("downloads", anime_slug)
    os.makedirs(download_dir, exist_ok=True)
    
    downloaded_files = []
    for ep in selected:
        ep_num = ep['episode']
        print_bot(f"\n--&gt; Stalking Episode {ep_num}...")
        m3u8_link = get_m3u8_link(anime_slug, ep_num)
        if not m3u8_link:
            print_bot(f"The prey escaped! Could not extract m3u8 for Ep {ep_num}. 💨")
            continue
            
        filepath = os.path.join(download_dir, f"{anime_slug}_E{ep_num}.mp4")
        if download_with_ytdlp(m3u8_link, filepath, str(ep_num)):
            downloaded_files.append((ep_num, filepath))
        time.sleep(2)
        
    merged_video = None
    if merge_episodes and len(downloaded_files) > 1:
        ordered_files = [f for _, f in downloaded_files]
        first_ep = downloaded_files[0][0]
        last_ep = downloaded_files[-1][0]
        merged_video = merge_videos(ordered_files, anime_slug, season_number, str(first_ep), str(last_ep))
        if merged_video and not keep_individual_files:
            print_bot("Incinerating the remnants (cleaning up individual files)... 🔥")
            for f in ordered_files:
                try:
                    os.remove(f)
                except Exception:
                    pass

    # Handle Uploads
    files_to_upload = []
    if upload_merged_only and merged_video:
        files_to_upload = [merged_video]
    elif merged_video:
        files_to_upload = [merged_video]
        if keep_individual_files:
            files_to_upload.extend([f for _, f in downloaded_files if os.path.exists(f)])
    else:
        files_to_upload = [f for _, f in downloaded_files if os.path.exists(f)]
        
    if upload_destination != "None (Keep Local)" and files_to_upload:
        for path in files_to_upload:
            if upload_destination in ["GoFile.io Only", "Both Services"]:
                upload_to_gofile(path)
            if upload_destination in ["Google Drive Only", "Both Services"]:
                upload_to_gdrive(path)

    print_bot("=========================================================")
    print_bot("🎉 RITUAL COMPLETE! The souls are yours! Mwahaah! 😈🥂")
    print_bot("=========================================================")

try:
    main()
except Exception as e:
    print_bot(f"A fatal curse struck the script! {e} 💀")
